# Chapter 22: The case review

The notebook version of [`README.md`](README.md) and [`run.py`](run.py), generated by `make notebooks`. Run the cells in order. The lesson stops to ask for your predictions; answer in the box under the cell.

In [ ]:
# Setup: tell LANTERN which chapter this is, and give the lesson code the __file__
# a script has (the kernel runs in this lesson's folder).
import os
os.environ["LANTERN_CHAPTER"] = "22"
__file__ = os.path.abspath("run.py")

## The scene

Thursday 6 August, 17:30. The evening before Professional Standards report. Varma's whiteboard
list is the one she makes at the end of every case file.

Summaries of summaries. Memory nobody can audit. A prompt that only grows. A context tuned
without a baseline. A cache broken by a date. A fact with no episode.

"Each of these happened here," she says, "or it comes off the board."

Okafor has the recorded runs. You go down the list with him, one line at a time, and for each
you find the chapter and the number.

One line stays on the board with nothing against it but a question mark. Who else can write to
this machine at night.

## The concept

Most context failures do not fail loudly. A summary written over a false entry repeats it in
other words; a prompt gains a rule after every failure; a fact sits in memory with nothing
behind it. Each makes answers slightly worse for months, and each is easy to list and hard to
see. A list of pitfalls is only worth something when every line on it is tied to a run in
which it happened, with the number.

One line on the board needs a run of its own: **a context tuned without a baseline**. The
sharpest form of it is letting a program rewrite the context (amendment A1, placed here by
decision 0004). An optimiser proposes a revision of the standing orders, scores it on a set
of questions, and keeps it if the score goes up. Scored on the questions it was shown, it
always looks better. Only a held-out split, questions the proposer never saw, says whether it
is.

```mermaid
flowchart LR
    Q[26 signed questions] --> S{split by slice, seed 22}
    S --> T[training: 10]
    S --> H[held out: 8, never shown]
    O[standing orders] --> P[propose: the model revises from the training failures]
    T --> P
    P --> C[score on training]
    C -->|passes more| K[keep as the best]
    C -->|not more| D[drop]
    K --> P
    K --> F[score the best once on held out]
    H --> F
```

## How it works

1. **The board, against the record.** Each line is matched to the chapter that measured it,
   and the number is read from that chapter's recorded run in `.progress/results/`, not run
   again. A line with no recorded run is shown as "not recorded", with the command that would
   record it. No model.
2. **The split.** `optimise.split` takes two questions of every slice for training and two
   more held out, by a seeded shuffle of each slice: 10 and 8. The two rules-in-conflict
   questions both go to training, so the held-out score says nothing about that slice. No
   model.
3. **Propose, score, keep.** `optimise.optimise` runs three rounds. Each round the model
   (qwen3:8b) is shown the best standing orders so far and the training questions they
   failed, each with LANTERN's answer and what the grader needed, and writes a revision
   (`optimise.propose`). The revision is scored on the training split with `baseline.score`
   and kept only if it passes more; a revision with fewer than five numbered rules, or the
   same numbered rules as any earlier round (spacing and title aside), is not scored. The
   proposer samples at temperature 0.7 with its own seed each round (22, 23 and 24), and is
   told the rules it already tried that were not kept. Then the best is scored once on the held-out split, beside
   today's standing orders and chapter 04's rewrite by hand. Where chapter 02's baseline and
   chapter 04's rewrite were recorded on this machine (`.scratch/02/baseline.json`,
   `.scratch/04/rewrite.json`) their results are read, not asked again: the same questions,
   seed and context.
4. **What it learned, and what it dropped.** The rules the best revision added or reworded;
   any answer from the training questions it wrote into the rules (`optimise.leaked`);
   whether "not in evidence" survived; how many held-out questions the proposer was shown
   (zero, unless a challenge changes it).
5. **The line that stays.** Who else can write to this machine at night: the runs name the
   account and the machine, and no check here can name the person.

The loop in code is short, and the one line that matters is `kept`:

```python
for n in range(1, rounds + 1):
    failed = [r for r in best["results"] if not r["passed"]]
    candidate = propose(best["orders"], failed)
    results = score(candidate)                     # the training split only
    kept = sum(r["passed"] for r in results) > best["passed"]
```

## The lesson

The code from `run.py`, one cell per step.

In [ ]:
import json
import re
from dataclasses import replace

from lantern import baseline, case, context, learn, optimise
from lantern.config import ROOT
from lantern.llm import chat
from lantern.ui import Table, chapter, console, evidence, finding, note, save_shot, step

chapter("22", "The case review", """
Thursday 6 August, 17:30. The evening before Professional Standards report. Varma's whiteboard
list is the one she makes at the end of every case file.

Summaries of summaries. Memory nobody can audit. A prompt that only grows. A context tuned
without a baseline. A cache broken by a date. A fact with no episode.

"Each of these happened here," she says, "or it comes off the board."

Okafor has the recorded runs. You go down the list with him, one line at a time, and for each
you find the chapter and the number.

One line stays on the board with nothing against it but a question mark. Who else can write to
this machine at night.
""")

MINE = learn.mine()  # --mine: your split() and optimise() from practice/exercise.py
SPLIT = MINE.split if MINE else optimise.split
OPTIMISE = MINE.optimise if MINE else optimise.optimise
OUT = ROOT / ".scratch" / "22"
RESULTS = ROOT / ".progress" / "results"
ROUNDS = 3  # candidates the proposer writes; each is scored on the training split
TRAIN_PER_SLICE, HELD_PER_SLICE, SPLIT_SEED = 2, 2, 22
# Challenge: what the proposer is shown. "train" is the loop as it should be; "train+held"
# also shows it the held-out questions the standing orders failed.
PROPOSER_SEES = "train"
# Challenge: words every revision must keep, e.g. ("not in evidence",).
MUST_KEEP: tuple[str, ...] = ()
REWRITE = "rebuilt/standing-orders.md"  # chapter 04's rewrite by hand


def recorded(chapter_no: str) -> dict | None:
    path = RESULTS / f"{chapter_no}.json"
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return None


def line(key: str, chapter_no: str, say) -> dict:
    r = recorded(chapter_no)
    if r is None:
        return {"pitfall": key, "chapter": chapter_no, "shows": None,
                "when": "not recorded: run make lesson N=" + chapter_no}
    try:
        shows = say(r["metrics"])
    except (KeyError, TypeError, ZeroDivisionError):
        shows = None
    return {"pitfall": key, "chapter": chapter_no, "shows": shows,
            "when": r.get("at", "")[:10] if shows else "recorded, without this number"}


BOARD = [
    line("Summaries of summaries", "18", lambda m:
         f"{m['built_on_eighth']} of {m['summaries']} summaries were built on the eighth entry; "
         f"after it was retracted {m['old_summaries_say_cleared']} still said cleared, and "
         f"the Pellmar questions passed {m['passed_retracted_old_summaries']} of {m['asked']} "
         f"until they were rebuilt ({m['passed_retracted_rebuilt']} after)"),
    line("Memory nobody can audit", "20", lambda m:
         f"{m['unsourced']} of {m['entries']} entries rest on nothing; only the write log "
         f"says who wrote them: all {m['unsourced_by_service']} through the HQ sync account"),
    line("A prompt that only grows", "04", lambda m:
         f"{m['versions']} versions; {m['first_tokens']} tokens in March, {m['tokens']} now, "
         f"{m['rules']} rules, {m['lint_pairs']} pairs that disagree"),
    line("A context tuned without a baseline", "02", lambda m:
         f"the baseline is {m['passed']} of {m['asked']}, 95% interval "
         f"{m['interval_low_pct']} to {m['interval_high_pct']}%: a few questions either way "
         "is inside it (and step 3 of this chapter)"),
    line("A cache broken by a date", "16", lambda m:
         f"a change at the start of the context: {m['start_s']} s to read it again, against "
         f"{m['same_s']} s for the same context unchanged"),
    line("A fact with no episode", "12", lambda m:
         f"{m['unsupported']} of {m['entries']} entries name neither a record nor an episode"),
    line("Who else can write to this machine at night", "20", lambda m:
         f"{m['unsourced_by_service']} writes through svc-lantern-sync from hq-sync-01 at "
         "01:58; who administers that account is not in any run"),
]
LEFT_TO_A_PERSON = BOARD[-1]["pitfall"]

OUT.mkdir(parents=True, exist_ok=True)

### 1. The board, against the record

In [ ]:
step("1. The board, against the record")
p_board = learn.predict(
    f"Varma's board has {len(BOARD)} lines. Each stays only if a recorded run of this lab "
    "shows it happened here. How many have a number behind them?",
    ["Fewer than four", "Four to six", "All of them"])
table = Table(title="Each line on the board, and the recorded run that shows it")
for col in ("pitfall", "chapter", "what the recorded run shows", "recorded"):
    table.add_column(col)
for b in BOARD:
    table.add_row(b["pitfall"], b["chapter"], b["shows"] or "(no number)", b["when"])
console.print(table)
shown = [b for b in BOARD if b["shows"]]
missing = [b for b in BOARD if not b["shows"]]
without = ", ".join(f"{b['pitfall']} (chapter {b['chapter']})" for b in missing)
note(f"{len(shown)} of {len(BOARD)} lines have a recorded number behind them"
     + (f"; without one: {without}" if missing else "")
     + ". The numbers are read from .progress/results/, not run again.")
learn.reveal(p_board, answer=learn.bucket(len(shown), [3, len(BOARD) - 1]),
             why="A pitfall list copied from elsewhere is a list of other people's mistakes. "
             "This one is read from the unit's own runs, and a line with no run behind it is "
             "a line to run, not a line to believe.")

### 2. A program that rewrites the standing orders: the split

In [ ]:
step("2. A program that rewrites the standing orders: the split")
qs = baseline.questions()
qmap = {q["id"]: q for q in qs}
train_ids, held_ids = SPLIT(qs, train=TRAIN_PER_SLICE, held=HELD_PER_SLICE, seed=SPLIT_SEED)
table = Table(title=f"Questions by slice: training and held out (seed {SPLIT_SEED})")
for col in ("slice", "training", "held out", "not used"):
    table.add_column(col)
for sl in dict.fromkeys(q["slice"] for q in qs):
    ids = [q["id"] for q in qs if q["slice"] == sl]
    table.add_row(sl, ", ".join(i for i in ids if i in train_ids),
                  ", ".join(i for i in ids if i in held_ids),
                  ", ".join(i for i in ids if i not in train_ids + held_ids))
console.print(table)
note(f"{len(train_ids)} training questions, {len(held_ids)} held out, no question in both "
     f"({len(set(train_ids) & set(held_ids))} shared). The proposer is shown the training "
     f"failures only{'' if PROPOSER_SEES == 'train' else ', and here the held-out ones too'}.")

parts = context.sections()
start = case.read("state/standing-orders.md")


def with_orders(text: str):
    return [replace(s, text=text) if s.name == "standing orders" else s for s in parts]


def from_record(path: str, ids: list[str]) -> list | None:
    """Results recorded by an earlier chapter for the same questions, seed and context."""
    try:
        kept = {r["id"]: r for r in json.loads((ROOT / path).read_text(encoding="utf-8"))}
    except (OSError, ValueError):
        return None
    return [kept[i] for i in ids] if all(i in kept for i in ids) else None


def score_on(ids: list[str]):
    return lambda orders: baseline.score(with_orders(orders), chat=chat, only=ids)


reused = []
base_train = from_record(".scratch/02/baseline.json", train_ids)
base_held = from_record(".scratch/02/baseline.json", held_ids)
if base_train is None or base_held is None:
    base_train, base_held = score_on(train_ids)(start), score_on(held_ids)(start)
else:
    reused.append("chapter 02's baseline")

prompts: list[str] = []
tried: list[str] = []  # every revision written so far; the proposer is told not to repeat them
seeds = iter(range(SPLIT_SEED, SPLIT_SEED + ROUNDS))


def proposer_chat(prompt, **kw):
    prompts.append(prompt)
    return chat(prompt, **kw)


def propose(orders: str, failed: list) -> str:
    shown_failed = optimise.failures(failed, qmap)
    if PROPOSER_SEES != "train":
        shown_failed += optimise.failures(base_held, qmap)
    out = optimise.propose(orders, shown_failed, chat=proposer_chat, seed=next(seeds),
                           must_keep=MUST_KEEP, tried=tuple(tried))
    tried.append(out)
    return out

### 3. Propose, score, keep: on the training split only

In [ ]:
step("3. Propose, score, keep: on the training split only")
p_held = learn.predict(
    f"The loop writes {ROUNDS} revisions and keeps one only if it passes more of the "
    f"{len(train_ids)} training questions than the best so far. On the {len(held_ids)} "
    "questions it never saw, will the best revision do better than today's standing orders?",
    ["Better", "The same", "Worse"])
history = OPTIMISE(start, propose=propose, score=score_on(train_ids), rounds=ROUNDS,
                   start_results=base_train)
table = Table(title="The loop, round by round (training split)")
for col in ("round", "rules", "words", "passed (training)", "decision"):
    table.add_column(col)
for h in history:
    table.add_row(str(h["round"]), str(optimise.rules(h["orders"])),
                  str(len(h["orders"].split())),
                  "" if h["passed"] is None else f"{h['passed']} of {len(train_ids)}", h["why"])
console.print(table)
top = [h for h in history if h["kept"]][-1]
best_orders = top["orders"]
best_train = top["passed"]
scored = sum(h["passed"] is not None for h in history[1:])  # revisions that were not copies
if top["round"] == 0:
    best_held = base_held
else:
    best_held = score_on(held_ids)(best_orders)
rewrite_held = from_record(".scratch/04/rewrite.json", held_ids)
if rewrite_held is None:
    rewrite_held = score_on(held_ids)(case.read(REWRITE))
else:
    reused.append("chapter 04's rewrite")
k = {"base_train": sum(r["passed"] for r in base_train),
     "base_held": sum(r["passed"] for r in base_held),
     "best_held": sum(r["passed"] for r in best_held),
     "rewrite_held": sum(r["passed"] for r in rewrite_held)}
table = Table(title="Training against held out")
for col in ("standing orders", f"training ({len(train_ids)})", f"held out ({len(held_ids)})"):
    table.add_column(col)
table.add_row("as they are (13 July)", str(k["base_train"]), str(k["base_held"]))
table.add_row(f"the loop's best (round {top['round']})", str(best_train), str(k["best_held"]))
table.add_row("chapter 04's rewrite, by hand", "", str(k["rewrite_held"]))
console.print(table)
train_gain, held_gain = best_train - k["base_train"], k["best_held"] - k["base_held"]
diff = baseline.changed(base_held, best_held)
note(f"On training the best gained {train_gain:+d}; held out it gained {held_gain:+d} "
     f"(fixed {', '.join(diff['fixed']) or 'none'}; broken {', '.join(diff['broken']) or 'none'})."
     + (f" Reused, not asked again: {' and '.join(reused)} (the same questions, seed and "
        "context)." if reused else ""))
learn.reveal(p_held, answer=0 if held_gain > 0 else 1 if held_gain == 0 else 2,
             why="The loop only ever sees the training score, so a revision that fits those "
             f"{len(train_ids)} questions is kept whether or not it helps the rest. The "
             "held-out score is the only one that says which.")

### 4. What it learned, and what it dropped

In [ ]:
step("4. What it learned, and what it dropped")
p_nie = learn.predict(
    "Rule 2 of the standing orders says: cite the evidence, or say \"not in evidence\". Does "
    "the loop's best revision still contain those words?", ["Yes", "No"])
leaked_train = optimise.leaked(best_orders, start, [qmap[i] for i in train_ids])
leaked_held = optimise.leaked(best_orders, start, [qmap[i] for i in held_ids])


def numbered(text: str) -> list[str]:
    return [re.sub(r"^\d+\.\s*", "", x).strip() for x in text.splitlines()
            if re.match(r"\d+\.", x)]


old_rules = set(numbered(start))
new_rules = numbered(best_orders)
added = [r for r in new_rules if r not in old_rules]
keeps = baseline.NOT_IN_EVIDENCE in best_orders.lower()
evidence(f"The best revision (round {top['round']}): rules it added or reworded, first six",
         "\n".join(f"- {r}" for r in added[:6]) or "(none: the loop kept nothing)")
note(f"{optimise.rules(start)} rules before, {optimise.rules(best_orders)} after; "
     f"{len(added)} new or reworded. Answers to the training questions written into it: "
     f"{', '.join(leaked_train) or 'none'}; to held-out questions: "
     f"{', '.join(leaked_held) or 'none'}. \"not in evidence\": "
     f"{'kept' if keeps else 'gone'}.")
saw_held = sum(any(qmap[i]["question"] in p for p in prompts) for i in held_ids)
note(f"Held-out questions shown to the proposer: {saw_held} of {len(held_ids)}.")
learn.reveal(p_nie, answer=0 if keeps else 1,
             why="The proposer was asked to fix failures, not to keep rules. A rule that "
             "protects questions it was not shown has nobody speaking for it in the loop, so "
             "keeping it is either luck or a constraint someone wrote down.")
(OUT / "history.json").write_text(json.dumps({
    "train": train_ids, "held": held_ids, "history": history, "best_held": best_held,
    "base_held": base_held, "rewrite_held": rewrite_held, "prompts": prompts}, indent=2),
    encoding="utf-8")

### 5. The line that stays on the board

In [ ]:
step("5. The line that stays on the board")
night = BOARD[-1]
note(f"{night['pitfall']}: {night['shows'] or 'chapter 20 has not been run here'}. No check "
     "in this lab can say who sits behind a service account. That is left to a person, and "
     "it is written down as left to a person.")

learn.record(
    pitfalls=len(BOARD),
    pitfalls_recorded=len(shown),
    train=len(train_ids),
    held=len(held_ids),
    rounds=ROUNDS,
    candidates_scored=sum(h["passed"] is not None for h in history[1:]),
    kept=sum(h["kept"] for h in history[1:]),
    best_round=top["round"],
    baseline_train=k["base_train"],
    best_train=best_train,
    baseline_held=k["base_held"],
    best_held=k["best_held"],
    rewrite_held=k["rewrite_held"],
    train_gain=train_gain,
    held_gain=held_gain,
    held_fixed=len(diff["fixed"]),
    held_broken=len(diff["broken"]),
    leaked_train=len(leaked_train),
    leaked_held=len(leaked_held),
    keeps_not_in_evidence=keeps,
    start_rules=optimise.rules(start),
    best_rules=optimise.rules(best_orders),
    proposer_saw_held=saw_held,
    revisions_scored=scored,
    must_keep=len(MUST_KEEP),
    reused_recorded=len(reused),
    mine=MINE is not None,
)

if scored == 0:
    LESSON = ("No revision was scored, so this run shows nothing about tuning; the loop needs "
              "a proposer that writes something new.")
elif train_gain > held_gain:
    LESSON = (f"It gained {train_gain} on the questions it saw and {held_gain} on those it did "
              "not: a context tuned on what it is scored against looks better than it is, and "
              "the held-out split is the baseline that says so.")
else:
    LESSON = (f"It gained {train_gain} on the questions it saw and {held_gain} on those it did "
              "not; on this run the held-out split did not fall behind, and only the held-out "
              "split could say so.")
finding(f"""
{len(shown)} of the {len(BOARD)} lines on Varma's board have a recorded run of this lab
behind them. A program asked to rewrite the standing orders wrote {ROUNDS} revisions, scored
{scored} of them on {len(train_ids)} training questions (the others repeated rules already
tried), and kept the one from round {top['round']}: on
training {k['base_train']} became {best_train}, held out {k['base_held']} became
{k['best_held']} of {len(held_ids)}, where chapter 04's rewrite by hand passes
{k['rewrite_held']}. It wrote {len(leaked_train)} answer(s) from the cases it saw into the
orders, and it {'kept' if keeps else 'dropped'} "not in evidence". {LESSON} The
last line, {LEFT_TO_A_PERSON.lower()}, is left to a person.
""")
save_shot("22-common-pitfalls")

## The nuances

- **Every line on the board has a number behind it.** 7 of 7 lines matched a recorded run of
  this lab, read from `.progress/results/`, not run again. A pitfall list copied from
  elsewhere is a list of other people's mistakes; this one is the unit's own. The chapter 16
  line is that chapter's latest recorded run (13:46 to 13:55).
- **The overfitting beat did not show on this run, because no revision beat the orders.**
  Round 1 returned the same numbered rules as round 0 and was not scored. Round 2 added three
  rules and passed 7 of 10, the same as the orders, so it was dropped; round 3 added two and
  passed 6. The best stayed round 0, the standing orders as they are, and its held-out score
  is chapter 02's baseline on those questions: 4 of 8, none fixed and none broken. A loop can
  only overfit through a revision it keeps.
- **The proposer wrote at the failures it was shown.** The three training failures were Q-20,
  Q-24 and Q-26. Round 2's new rules were "If a tool is not connected, say "not in evidence"",
  "State amounts without currency symbols" and "Use the full names of entities, including
  Ltd, and do not rephrase them": rules aimed at Q-24's answer about an unconnected tool and
  Q-26's grader, which forbids a currency and a rephrased name. None of the three moved a
  training question.
- **A new rule can break what it was not written for.** Round 3 added the first two of those
  rules and lost Q-10, which the orders passed: LANTERN answered "The tool to look up company
  information is not connected in this session. Nothing was checked." for a question the case
  memory answers. That is one question in ten, the kind of move chapter 02's interval (46 to
  81% on 26 questions) says is not a finding either way.
- **Why the held-out split is the evidence.** The loop keeps whatever scores more on the ten
  questions it is scored on, and the proposer reads what each failed answer had to contain.
  A revision that wins there can win by fitting those ten, even by writing their answers into
  the rules; only questions the proposer never saw can say whether it is better. Here the
  proposer was shown 0 of the 8 held-out questions, and no answer from the training questions
  was written into any rule.
- **An earlier run scored copies.** Before the fix, every revision was the orders with their
  title gone and trailing spaces, and the loop scored each because it compared the bytes; it
  now compares the numbered rules with every earlier round.
- **Small splits move in whole questions.** On 8 held-out questions one answer is 12.5 points.
  The two rules-in-conflict questions both went to training, so the held-out score says
  nothing about that slice.
- **The hand rewrite has seen more than the loop.** Chapter 04's rewrite, 5 of 8 held out, was
  written from the baseline's failures over all 26 questions, so on the held-out questions it
  is not a clean comparison either.

## Try this

1. Find a pitfall in your own runs that is not on the list.
2. Pick one remedy and remove it. Which chapter's number moves?
3. Run the loop for six rounds instead of three. Does the training score keep rising, and the
   held-out score with it?

## Master it

After this chapter you can tie each pitfall to a recorded run, split a question set so a
tuning loop cannot see half of it, write the loop of propose, score and keep, and say why its
own score is not the evidence.

**Predict, then see.** `make lesson N=22` stops three times: how many lines on the board have a
number behind them, whether the loop's best does better on the questions it never saw, and
whether its best revision still says "not in evidence".

**Build it.** Write `split` and `optimise` in [`practice/exercise.py`](practice/exercise.py):

```bash
make exercise N=22          # 10 offline tests
make hint N=22
make lesson N=22 ARGS=--mine
```

**Check yourself.** `make quiz N=22`: seven questions.

**Break it.** `make challenge N=22`: your loop on the case; the proposer shown the held-out
questions; a rule the loop may not drop; and six rounds instead of three.

`make progress` shows every chapter.

The cells below run the same commands inside this notebook.

### Check yourself

In [ ]:
%run ../../scripts/learn.py quiz 22

### Build it

Open [`practice/exercise.py`](practice/exercise.py) beside this notebook, fill in the TODOs, save, and run the grader. Hints come one at a time.

In [ ]:
%run ../../scripts/learn.py exercise 22

In [ ]:
%run ../../scripts/learn.py hint 22

### Break it

The challenges are checked against your latest run of the lesson cells above (or of `make lesson`). Change the code, rerun the lesson, then check.

In [ ]:
%run ../../scripts/learn.py challenge 22

In [ ]:
%run ../../scripts/learn.py progress

## What the evidence now shows

Every line on Varma's board is tied to a recorded run of this lab, 7 of 7, and the last of
them, who else can write to this machine at night, is written down as left to a person. A
program asked to rewrite the standing orders wrote three revisions; one repeated the orders'
rules and was not scored, one passed 7 of 10 training questions and one 6, against the orders'
7, so neither was kept. The best stayed the standing orders as they are, 4 of 8 held out, where
chapter 04's rewrite by hand passes 5. Its new rules aimed at the failures it was shown and
one of them broke a question it was not aimed at. On this run tuning gained nothing on either
split; had it gained on training, only the held-out split could have said whether the gain
was real.

**Next:** [Chapter 23: Friday 7 August](../23-context-in-production/README.md)